In [1]:
import os
import json
import uuid
import pandas as pd
import numpy as np
import chromadb

class VectorDatabaseManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="b2b_insights"):
        # 1. Define the basic settings
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        
        # 2. Declare the empty variables (Best Practice)
        self.client = None
        self.collection = None
        
        # 3. Boot up the database immediately upon creation
        self._initialize_store()

    def _initialize_store(self):
        """Creates the folder and connects to ChromaDB."""
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # Security Guard: The Client connects to the folder
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        # Filing Cabinet: The Collection where data lives
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "B2B Market Insights Vector Store"}
        )
        print(f"[INFO] Initialized vector store with collection: '{self.collection_name}'")

    def upsert_data(self, csv_path, embeddings_path):
        """Loads data from CSV/Numpy and pushes it into the Collection."""
        print("[INFO] Loading data and embeddings into memory...")
        df = pd.read_csv(csv_path)
        
        # Load the numpy array of 384-dimensional vectors
        embeddings_array = np.load(embeddings_path)
        
        # ChromaDB requires lists for insertion
        ids = []
        documents = []
        metadatas = []
        embeddings = []

        print("[INFO] Formatting data for ChromaDB. Generating UUIDs...")
        for index, row in df.iterrows():
            # 1. Create a perfectly unique ID using the uuid library
            unique_id = str(uuid.uuid4())
            ids.append(unique_id)
            
            # 2. The Document is the main text content
            documents.append(str(row['content']))
            
            # 3. The Embeddings (converting Numpy array row to a standard Python list)
            embeddings.append(embeddings_array[index].tolist())
            
            # 4. The Metadata (Converting string back to a real dictionary)
            metadata_dict = json.loads(row['metadata'])
            
            # Ensure the title is saved in the metadata
            metadata_dict['title'] = str(row['title']) 
            
            metadatas.append(metadata_dict)

        print(f"[INFO] Upserting {len(ids)} records into the database. This may take a moment...")
        
        # Upload in batches of 1000 to prevent memory crashes
        batch_size = 1000
        for i in range(0, len(ids), batch_size):
            self.collection.add(
                ids=ids[i : i + batch_size],
                embeddings=embeddings[i : i + batch_size],
                documents=documents[i : i + batch_size],
                metadatas=metadatas[i : i + batch_size]
            )
            print(f"       -> Uploaded batch {i} to {min(i + batch_size, len(ids))}...")
            
        print(f"[INFO] Success! Vector DB now holds {self.collection.count()} total records.")



if __name__ == "__main__":

    db_manager = VectorDatabaseManager()
    
    # 2. Define the exact paths from your previous steps
    CSV_FILE = "data/standardized_data.csv"
    EMBEDDINGS_FILE = "embeddings/embeddings.npy"
    
    # 3. Run the insertion process
    if os.path.exists(CSV_FILE) and os.path.exists(EMBEDDINGS_FILE):
        db_manager.upsert_data(csv_path=CSV_FILE, embeddings_path=EMBEDDINGS_FILE)
    else:
        print("[ERROR] Could not find the CSV or Embeddings file. Please check your paths.")

[INFO] Initialized vector store with collection: 'b2b_insights'
[INFO] Loading data and embeddings into memory...
[INFO] Formatting data for ChromaDB. Generating UUIDs...
[INFO] Upserting 8469 records into the database. This may take a moment...
       -> Uploaded batch 0 to 1000...
       -> Uploaded batch 1000 to 2000...
       -> Uploaded batch 2000 to 3000...
       -> Uploaded batch 3000 to 4000...
       -> Uploaded batch 4000 to 5000...
       -> Uploaded batch 5000 to 6000...
       -> Uploaded batch 6000 to 7000...
       -> Uploaded batch 7000 to 8000...
       -> Uploaded batch 8000 to 8469...
[INFO] Success! Vector DB now holds 8469 total records.


In [5]:
results = db_manager.collection.peek(limit=10)


for i in range(len(results['ids'])):
    print(f"--- Chunk {i+1} ---")
    print(f"ID: {results['ids'][i]}")
    print(f"Document: {results['documents'][i][:2000]}...")  # Showing first 200 characters of the content
    print(f"Metadata: {results['metadatas'][i]}")
    print("-" * 40 + "\n")

--- Chunk 1 ---
ID: 6eac5773-076f-45ce-a086-497e1c3e618a
Document: I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists. Product setup Technical issue...
Metadata: {'Date of Purchase': '2021-03-22', 'Ticket Channel': 'Social media', 'Customer Gender': 'Other', 'Customer Age': 32, 'title': 'GoPro Hero'}
----------------------------------------

--- Chunk 2 ---
ID: 7fe66d60-f225-488e-afa9-03e3a762f1a7
Document: I'm having an issue with the {product_purchased}. Please assist.

If you need to change an existing product.

I'm having an issue with the {product_purchased}. Please assist.

If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly. Peripheral compatibility Technical issue...
Metadata: {'Ticket Channel': 